In [ ]:
import sys
sys.path.append("../")

#
# TO BE USED WITH the Godot project in ../ackermann
#

from lib.system.cart import *
from lib.dds.dds import *
from lib.utils.time import *
from lib.system.controllers import *
from lib.data.dataplot import *
from lib.system.polar import *
from manipulator_control import *
from lib.system.trajectory import *

dds = DDS()
dds.start()

dds.subscribe(['tick', 'TerrainHeight', 'TerrainSlope'])

arm = FourJointsManipulatorControl()
terrain = MeasuredTerrainProfile()
cart2d = AckermannSlopeLoad(
    cart_mass=10,
    friction=0.8,
    wheel_radius=0.5,
    wheelbase=2.0,
    terrain=terrain,
)

linear_speed_controller = PID_Controller(50.0, 10.0, 0, 100.0)
polar_position = Polar2DController(1.0, 3.0, # kp = 1, vmax = 3 m/s
                                   5.0, math.radians(40)) # kp = 5, alpha_max = 40 deg

virtual_robot = StraightLine2DMotion(1.0, 1.5, 1.5)

# Edit these coordinates and the radius to configure the pickup/delivery task.
point_a = (0.0, 0.0)
point_b = (0, -25)
zone_radius = 0.25
arrival_threshold = zone_radius
packages_to_load = [5.0, 5.0, 5.0]

cart2d.set_pose(point_a[0], point_a[1])
load_zone = PackageTransferZone('A', point_a, zone_radius, 'load', packages_to_load)
unload_zone = PackageTransferZone('B', point_b, zone_radius, 'unload')
cargo_zones = [load_zone, unload_zone]
cargo_events = []

initial_load_event = load_zone.process(cart2d)
if initial_load_event is not None:
    cargo_events.append(initial_load_event)
    print('cargo_event:', initial_load_event)

virtual_robot.start_motion(point_a, point_b)

vdp = DataPlotter()
vdp.set_x("time (seconds)")
vdp.add_y("target_speed", "Target Linear Speed")
vdp.add_y("current_speed", "Current Linear Speed")

sdp = DataPlotter()
sdp.set_x("time (seconds)")
sdp.add_y("angle", "Steering Angle")

cdp = DataPlotter()
cdp.set_x("time (seconds)")
cdp.add_y("payload", "Payload Mass (kg)")
cdp.add_y("height", "Height (m)")
cdp.add_y("slope", "Slope (degrees)")


t = Time()
t.start()
while t.get() < 25:

    dds.wait('tick')
    delta_t = t.elapsed()
    terrain_height = dds.read('TerrainHeight')
    terrain_slope = dds.read('TerrainSlope')
    if terrain_height is not None and terrain_slope is not None:
        terrain.update(cart2d.s, terrain_height, terrain_slope)
    
    (target_x, target_y) = virtual_robot.evaluate(delta_t)
    pose = cart2d.get_pose()

    (target_speed, steering_angle) = polar_position.evaluate(delta_t, target_x, target_y, pose)

    (v, w) = cart2d.get_speed()
    torque = linear_speed_controller.evaluate(delta_t, target_speed - v)
    
    cart2d.evaluate(delta_t, torque, steering_angle)
    pose_3d = cart2d.get_pose_3d()

    for zone in cargo_zones:
        cargo_event = zone.process(cart2d)
        if cargo_event is not None:
            cargo_events.append(cargo_event)
            print('cargo_event:', cargo_event)

    mission_state = 9.0 if unload_zone.triggered else (5.0 if load_zone.triggered else 0.0)

    dds.publish('X', pose_3d[0], DDS.DDS_TYPE_FLOAT)
    dds.publish('Y', pose_3d[1], DDS.DDS_TYPE_FLOAT)
    dds.publish('Z', pose_3d[2], DDS.DDS_TYPE_FLOAT)
    dds.publish('Theta', pose_3d[3], DDS.DDS_TYPE_FLOAT)
    dds.publish('Slope', pose_3d[4], DDS.DDS_TYPE_FLOAT)
    dds.publish('PayloadMass', cart2d.get_payload_mass(), DDS.DDS_TYPE_FLOAT)
    dds.publish('MissionState', mission_state, DDS.DDS_TYPE_FLOAT)

    vdp.append_x(t.get())
    vdp.append_y("current_speed", v)
    vdp.append_y("target_speed", target_speed)
    
    sdp.append_x(t.get())
    sdp.append_y("angle", math.degrees(steering_angle))
    cdp.append_x(t.get())
    cdp.append_y("payload", cart2d.get_payload_mass())
    cdp.append_y("height", pose_3d[2])
    cdp.append_y("slope", math.degrees(pose_3d[4]))

vdp.plot()
sdp.plot()
cdp.plot()
final_pose = cart2d.get_pose_3d()
final_error = math.hypot(final_pose[0] - point_b[0], final_pose[1] - point_b[1])
reached = final_error <= arrival_threshold
delivery_complete = reached and load_zone.triggered and unload_zone.triggered and cart2d.get_payload_mass() == 0
print('A:', point_a, 'B:', point_b, 'zone_radius_m:', zone_radius)
print('cargo_events:', cargo_events)
print('final_pose:', final_pose, 'final_error_m:', final_error)
print('payload_kg:', cart2d.get_payload_mass(), 'destination_reached:', reached, 'delivery_complete:', delivery_complete)
dds.stop()
